In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.cube")
print("Cube schema ready.")

In [0]:

df_dim_store = spark.table("workspace.silver.store")

df_dim_store.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.dim_store")
print(f"cube.dim_store: {df_dim_store.count()} rows")
display(df_dim_store.limit(5))

In [0]:

df_dim_technician = (
    spark.table("workspace.silver.order")
    .filter(F.col("technician_id").isNotNull())
    .select("technician_id", "technician_name")
    .dropDuplicates(["technician_id"])
    .orderBy("technician_id")
)

df_dim_technician.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.dim_technician")
print(f"cube.dim_technician: {df_dim_technician.count()} rows")
display(df_dim_technician.limit(5))

In [0]:

df_dim_estimator = (
    spark.table("workspace.silver.estimate")
    .filter(F.col("estimator_id").isNotNull())
    .select("estimator_id", "estimator_name")
    .dropDuplicates(["estimator_id"])
    .orderBy("estimator_id")
)

df_dim_estimator.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.dim_estimator")
print(f"cube.dim_estimator: {df_dim_estimator.count()} rows")
display(df_dim_estimator.limit(5))

In [0]:

df_fact_order = (
    spark.table("workspace.silver.order")
    .select(
        "order_id",
        "store_id",                        # FK → dim_store
        "technician_id",                   # FK → dim_technician
        "service_type",
        "vehicle_no", "vehicle_make", "vehicle_model",
        "vehicle_in_datetime", "vehicle_out_datetime",
        "planned_work_start_datetime", "actual_work_start_datetime",
        "planned_completion_datetime", "actual_completion_datetime",
        "promised_delivery_datetime", "actual_delivery_datetime",
        "order_status",
        "customer_name", "customer_phone"
    )
)

df_fact_order.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.fact_order")
print(f"cube.fact_order: {df_fact_order.count()} rows")
display(df_fact_order.limit(5))

In [0]:

df_fact_survey = spark.table("workspace.silver.customer_survey")

df_fact_survey.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.fact_survey")
print(f"cube.fact_survey: {df_fact_survey.count()} rows")
display(df_fact_survey.limit(5))

In [0]:

df_fact_estimate = (
    spark.table("workspace.silver.estimate")
    .select(
        "estimate_id",
        "order_id",         
        "version_no",
        "estimate_amount",
        "created_at",
        "created_by",
        "estimate_type",
        "estimator_id"                   
    )
)

df_fact_estimate.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.fact_estimate")
print(f"cube.fact_estimate: {df_fact_estimate.count()} rows")
display(df_fact_estimate.limit(5))

In [0]:

df_fact_invoice = spark.table("workspace.silver.invoice")

df_fact_invoice.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.fact_invoice")
print(f"cube.fact_invoice: {df_fact_invoice.count()} rows")
display(df_fact_invoice.limit(5))

In [0]:

df_fact_budget = spark.table("workspace.silver.ns_budget")

df_fact_budget.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.cube.fact_budget")
print(f"cube.fact_budget: {df_fact_budget.count()} rows")
display(df_fact_budget.limit(5))